In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import scipy.io
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [2]:
# GPU 환경 확인
print("=" * 60)
print("[환경 확인]")
print(f"  CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU 이름: {torch.cuda.get_device_name()}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  VRAM 총량: {vram_total:.1f} GB")
    torch.cuda.empty_cache()
print("=" * 60)

[환경 확인]
  CUDA 사용 가능: True
  GPU 이름: NVIDIA GeForce RTX 4060
  VRAM 총량: 8.0 GB


In [3]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

In [4]:
# 베어링 결함 주파수 계산
class BearingFaultCalculator:
    def __init__(self, rpm, d_in, d_out, width):
        self.rpm = rpm
        self.fr = rpm / 60.0  # 회전 주파수 (Hz)
 
        # Pitch Diameter : (내경 + 외경) / 2
        self.pd = (d_in + d_out) / 2.0
 
        # NSK 6205 기준 볼 직경 : 약 7.9375mm (5/16 inch)
        self.bd = 7.9375
 
        # NSK 6205 기준 볼 개수: 9개
        self.n = 9
 
        # 접촉각: 깊은 홈 볼 베어링(Deep Groove Ball Bearing)은 0도
        self.angle = 0
        self.cos_angle = np.cos(np.deg2rad(self.angle))

    # 결함 주파수 계산
    def calculate_frequencies(self):
        geo_factor = (self.bd / self.pd) * self.cos_angle
 
        bpfo = (self.n / 2.0) * self.fr * (1 - geo_factor)
        bpfi = (self.n / 2.0) * self.fr * (1 + geo_factor)
        bsf = (self.pd / (2.0 * self.bd)) * self.fr * (1 - geo_factor ** 2)
        ftf = (1 / 2.0) * self.fr * (1 - geo_factor)
 
        return {
            "RPM": self.rpm,
            "Rotation (Hz)": self.fr,
            "BPFO (Outer Race)": bpfo,
            "BPFI (Inner Race)": bpfi,
            "BSF (Ball Spin)": bsf,
            "FTF (Cage)": ftf,
        }

In [5]:
# FFT 기반 베어링 결함 주파수 파워 분석
def calculate_bearing_power(signal, fs, fault_freq, margin_hz = 5.0, min_freq = 1000.0, max_freq = 3000.0):
    # DC 성분 제거 (평균값 차감)
    signal = signal - np.mean(signal)
 
    # FFT 계산 (단측 스펙트럼)
    n_samples = len(signal)
    fft_result = np.fft.fft(signal)
    fft_magnitude = np.abs(fft_result[: n_samples // 2])
 
    # 주파수 축
    freqs = np.fft.fftfreq(n_samples, d=1 / fs)[: n_samples // 2]
 
    # 파워 스펙트럼: 단측이므로 2를 곱하고 N²으로 정규화 (Parseval 정리)
    power_spectrum = 2.0 * (fft_magnitude ** 2) / (n_samples ** 2)
 
    # 분석 범위 내 하모닉들의 평균 파워 수집
    avg_powers = []
    for harmonic in range(1, 20):
        harmonic_freq = fault_freq * harmonic
 
        if min_freq <= harmonic_freq <= max_freq:
            freq_mask = (freqs >= harmonic_freq - margin_hz) & (
                freqs <= harmonic_freq + margin_hz
            )
            if np.any(freq_mask):
                region_power = np.mean(power_spectrum[freq_mask])
                avg_powers.append(region_power)
 
    return np.mean(avg_powers) if avg_powers else None

In [6]:
# RAG - FAISS 기반 유사 사례 검색

# RAG 데이터 캐시
_rag_cache = {
    "data": None,
    "index": None,
    "model": None,
    "embeddings": None,
}

# RAG 데이터를 로드하고 FAISS 인덱스를 생성
def load_rag_data(json_path = "KAIST_Bearing_report_varied_actions_ko_v2.json", model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                  use_gdrive = False, gdrive_file_id = None):
    import faiss
    from sentence_transformers import SentenceTransformer
 
    if _rag_cache["data"] is not None:
        return (
            _rag_cache["data"],
            _rag_cache["index"],
            _rag_cache["model"],
            _rag_cache["embeddings"],
        )

    # JSON 로드
    with open(json_path, "r", encoding="utf-8") as f:
        rag_data = json.load(f)
 
    # 임베딩 모델 로드
    print("임베딩 모델 로딩 중 (CPU)")
    embed_model = SentenceTransformer(model_name, device="cpu")
 
    # 검색용 텍스트 생성 및 임베딩
    search_texts = [
        f"{item.get('total', '')} {item.get('report', '')}" for item in rag_data
    ]
    embeddings = embed_model.encode(
        search_texts, show_progress_bar=True, convert_to_numpy=True
    )
 
    # L2 정규화 -> Inner Product = 코사인 유사도
    faiss.normalize_L2(embeddings)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
 
    print(f"RAG 데이터 로드 완료: {len(rag_data)}개 항목, 벡터 차원: {dimension}")
 
    # 캐시에 저장
    _rag_cache.update(
        data=rag_data, index=index, model=embed_model, embeddings=embeddings
    )
 
    return rag_data, index, embed_model, embeddings

In [19]:
# Faiss + 수치 비교
def search_similar_cases(load, bpfi_avg_power, bpfo_avg_power, rms, top_k = 3, faiss_candidates = 20, semantic_weight = 0.4, numerical_weight = 0.6):
    import faiss
    json_file_path = r".\KAIST_Bearing_report_varied_actions_ko_v2.json"
 
    rag_data, index, embed_model, embeddings = load_rag_data(
        json_path=json_file_path,
        use_gdrive=True
    )

    # 현재 측정 데이터를 자연어 쿼리로 변환
    bpfi_str = f"{bpfi_avg_power:.4e}" if bpfi_avg_power is not None else "측정 불가"
    bpfo_str = f"{bpfo_avg_power:.4e}" if bpfo_avg_power is not None else "측정 불가"
 
    query_text = (
        f"베어링 진동 분석 결과: "
        f"부하 {load}Nm, RMS {rms:.4f}, "
        f"BPFI 평균 파워 {bpfi_str}, "
        f"BPFO 평균 파워 {bpfo_str}"
    )
 
    print(f"FAISS 의미 검색 — 후보 {faiss_candidates}개 추출")
    print(f"  쿼리: {query_text}")
 
    # 쿼리 텍스트를 임베딩 벡터로 변환
    query_embedding = embed_model.encode(
        [query_text], convert_to_numpy=True
    )
    faiss.normalize_L2(query_embedding)  # L2 정규화 (코사인 유사도 계산용)
 
    # FAISS 검색 : 코사인 유사도 상위 N개 반환
    faiss_k = min(faiss_candidates, len(rag_data))
    scores, indices = index.search(query_embedding, faiss_k)
 
    print(f"-> 의미 유사도 범위: {scores[0][-1]:.4f} ~ {scores[0][0]:.4f}")



    # 수치 오차 기반 리랭킹
    print(f"수치 오차 리랭킹 — 상위 {top_k}개 선정")
 
    candidates = []
    for rank, (faiss_score, data_idx) in enumerate(zip(scores[0], indices[0])):
        item = rag_data[data_idx]
        item_bpfi = item.get("BPFI_평균_파워_1000_3000Hz")
        item_bpfo = item.get("BPFO_평균_파워_1000_3000Hz")
 
        # 수치 유사도 계산 : BPFI/BPFO 절대 오차 -> 유사도로 변환
        if item_bpfi is not None and item_bpfo is not None:
            bpfi_diff = (
                abs(float(item_bpfi) - float(bpfi_avg_power))
                if bpfi_avg_power is not None
                else float("inf")
            )
            bpfo_diff = (
                abs(float(item_bpfo) - float(bpfo_avg_power))
                if bpfo_avg_power is not None
                else float("inf")
            )
            total_error = bpfi_diff + bpfo_diff
            numerical_sim = 1.0 / (1.0 + total_error)
        else:
            bpfi_diff = float("inf")
            bpfo_diff = float("inf")
            total_error = float("inf")
            numerical_sim = 0.0
 
        # 의미 유사도 : FAISS 코사인 점수 (0~1 범위로 정규화)
        semantic_sim = max(0.0, (faiss_score + 1.0) / 2.0)
 
        # 최종 점수 : 가중 결합
        combined_score = (
            semantic_weight * semantic_sim
            + numerical_weight * numerical_sim
        )
 
        candidates.append(
            {
                "data": item,
                "combined_score": combined_score,
                "semantic_sim": semantic_sim,
                "numerical_sim": numerical_sim,
                "faiss_score": float(faiss_score),
                "total_error": total_error,
                "bpfi_diff": bpfi_diff,
                "bpfo_diff": bpfo_diff,
                "faiss_rank": rank + 1,
                "similarity": combined_score,
            }
        )

    # 최종 점수 기준 내림차순 정렬
    candidates.sort(key=lambda x: x["combined_score"], reverse=True)
    result = candidates[:top_k]
 
    # 검색 결과 출력
    if result:
        print(f"\n[검색 결과] 입력: BPFI={bpfi_avg_power:.2e}, BPFO={bpfo_avg_power:.2e}")
        for i, case in enumerate(result, 1):
            cd = case["data"]
            c_bpfi = cd.get("BPFI_평균_파워_1000_3000Hz")
            c_bpfo = cd.get("BPFO_평균_파워_1000_3000Hz")
            print(
                f"  [{i}] 오차합: {case['total_error']:.2e}, "
                f"유사도: {case['similarity']:.4f}"
            )
    else:
        print("⚠️ 유사한 케이스를 찾지 못했습니다.")
 
    return result

In [8]:
# RAG 컨텍스트 생성
def build_rag_context(similar_cases):
    if not similar_cases:
        return ""
 
    rag_context = "참고 사례:"
    for i, case in enumerate(similar_cases, 1):
        case_data = case["data"]
        bearing_state = case_data.get("베어링_상태", "N/A")
        report_all = case_data.get("report", "N/A")
 
        action_items = "N/A"
        if isinstance(report_all, str):
            split_key = "- 정비 액션 아이템:"
            if split_key in report_all:
                action_items = report_all.split(split_key, 1)[-1].strip()
 
        rag_context += f"\n[참고 사례 {i}] (유사도: {case['similarity']:.2f})\n"
        rag_context += f"- 베어링_상태: {bearing_state}\n"
        rag_context += f"- 정비 액션 아이템:\n{action_items}\n"
 
    return rag_context
 
# 진단 보고서 생성을 위한 프롬프트
def build_diagnosis_prompt(rms, bpfi_avg_power, bpfo_avg_power, rag_context):
    
    bpfi_str = f"{bpfi_avg_power:.2f}" if bpfi_avg_power is not None else "N/A"
    bpfo_str = f"{bpfo_avg_power:.2f}" if bpfo_avg_power is not None else "N/A"
 
    return f"""### 작성 지침
1. [참고 사례]가 제공된 경우, [참고 사례]의 '베어링_상태'를 바탕으로 '베어링_상태'를 판단해라.
2. [참고 사례] '정비 액션 아이템'을 바탕으로 '정비 액션 아이템'을 추천하라.
3. 현재 상태의 위험도를 RMS 수치와 BPFI(내륜결함 주파수) BPFO(외륜결함주파수) 평균 파워 값을 바탕으로 '정상', '주의', '위험' 중 하나로 평가해라.
4. 한글로 답변하고, '~함', '~임' 체로 작성해라.
{rag_context}
 
### 입력 데이터
- 진동 RMS 수치: {rms:.4f}
- BPFI 평균 파워 (1000~3000Hz): {bpfi_str}
- BPFO 평균 파워 (1000~3000Hz): {bpfo_str}
 
### 리포트
"""

In [9]:
# 진단 보고서 생성
def generate_kaist_report_qwen(stacked_data, metadata, fault_frequencies, fs, idx, text_generator, machine_name, use_rag):
    # 데이터 추출
    signal = stacked_data[idx, :]
    load = metadata.iloc[idx]["load"]
 
    # RMS 계산
    rms = np.sqrt(np.mean(signal ** 2))
 
    # BPFI/BPFO 평균 파워 계산
    bpfi_freq = fault_frequencies["BPFI (Inner Race)"]
    bpfo_freq = fault_frequencies["BPFO (Outer Race)"]
 
    bpfi_avg_power = calculate_bearing_power(
        signal, fs, bpfi_freq, margin_hz=5, min_freq=1000, max_freq=3000
    )
    bpfo_avg_power = calculate_bearing_power(
        signal, fs, bpfo_freq, margin_hz=5, min_freq=1000, max_freq=3000
    )
 
    # RAG 검색
    rag_context = ""
    if use_rag:
        try:
            similar_cases = search_similar_cases(
                load=load,
                bpfi_avg_power=bpfi_avg_power,
                bpfo_avg_power=bpfo_avg_power,
                rms=rms,
                top_k=2,
            )
            rag_context = build_rag_context(similar_cases)
        except Exception as e:
            print(f"RAG 검색 중 오류: {e}")
            rag_context = ""
 
    # 프롬프트 구성
    prompt = build_diagnosis_prompt(rms, bpfi_avg_power, bpfo_avg_power, rag_context)
 
    # 보고서 생성
    # Pipeline에 messages 형태로 전달하면 내부에서 chat template 적용
    messages = [
        {
            "role": "system",
            "content": (
                "당신은 베어링 진동 분석 전문가이다. "
                "아래 입력된 작성 지침을 바탕으로 '간결하고 명확한 조치 보고서'를 한글로 작성해라."
            ),
        },
        {"role": "user", "content": prompt},
    ]
 
    response = text_generator(messages)
    report = response[0]["generated_text"].strip()
 
    # 후처리
    for marker in ["### 리포트", "## 진단 보고서"]:
        if marker in report:
            report = report.split(marker)[-1].strip()
 
    return report, {
        "rms": rms,
        "bpfi_avg_power": bpfi_avg_power,
        "bpfo_avg_power": bpfo_avg_power,
        "load": load,
        "rpm": fault_frequencies["RPM"],
    }

In [10]:
# KAIST 데이터 로드 및 전처리
# 기존 예제 그대로 사용
def load_kaist_data(data_dir = "KAIST_Bearing", window_size = 25600 * 5, overlap = 0):
    mat_files = {
        "normal": f"{data_dir}/4Nm_Normal.mat",
        "BPFI_10": f"{data_dir}/4Nm_BPFI_10.mat",
        "BPFI_30": f"{data_dir}/4Nm_BPFI_30.mat",
        "BPFO_10": f"{data_dir}/4Nm_BPFO_10.mat",
        "BPFO_30": f"{data_dir}/4Nm_BPFO_30.mat",
    }
 
    all_data = []
    all_metadata = []
 
    for condition_name, file_path in mat_files.items():
        try:
            mat_data = scipy.io.loadmat(file_path)
 
            if "Signal" in mat_data:
                signal_struct = mat_data["Signal"][0, 0]
                y_values = signal_struct["y_values"].flatten()
                time_series_set = y_values["values"][0]
                time_series = time_series_set[:, 1]  # y_direction_housing_A
 
                # 구조화된 배열이면 숫자로 변환
                if time_series.dtype.names is not None:
                    time_series = np.array(
                        [float(x) for x in time_series.flatten()]
                    )
                else:
                    time_series = np.asarray(time_series, dtype=np.float64).flatten()
 
                # 세그먼트 분할
                stride = window_size - overlap
                first_offset = int(25600 * 0.5)  # 과도 응답 회피
                effective_length = len(time_series) - first_offset
                n_segments = (effective_length - window_size) // stride + 1
 
                for seg_idx in range(n_segments):
                    start = first_offset + seg_idx * stride
                    end = start + window_size
 
                    if end > len(time_series):
                        segment = np.zeros(window_size, dtype=np.float64)
                        segment[: len(time_series) - start] = time_series[start:]
                    else:
                        segment = time_series[start:end]
 
                    all_data.append(segment)
                    all_metadata.append(
                        {
                            "bearing_condition": condition_name,
                            "load": 4.0,
                            "segment_idx": seg_idx,
                            "filename": os.path.basename(file_path),
                        }
                    )
            else:
                # Signal 키가 없는 경우 fallback
                data_key = None
                for key in mat_data.keys():
                    if not key.startswith("__") and isinstance(mat_data[key], np.ndarray):
                        if mat_data[key].ndim >= 2:
                            data_key = key
                            break
 
                if data_key is None:
                    print(f"경고: {file_path}에서 데이터를 찾을 수 없습니다.")
                    continue
 
                data = np.asarray(mat_data[data_key], dtype=np.float64)
                if data.ndim == 1:
                    all_data.append(data)
                    all_metadata.append(
                        {"bearing_condition": condition_name, "load": 4.0}
                    )
                elif data.ndim == 2:
                    axis = 0 if data.shape[0] > data.shape[1] else 1
                    for i in range(data.shape[axis]):
                        sample = data[i, :] if axis == 0 else data[:, i]
                        all_data.append(sample)
                        all_metadata.append(
                            {"bearing_condition": condition_name, "load": 4.0}
                        )
 
        except Exception as e:
            print(f"오류: {file_path} 로드 실패 — {e}")
            continue
 
    if not all_data:
        raise ValueError("로드된 데이터가 없습니다.")
 
    # 모든 샘플을 동일한 길이로 정렬
    stacked_data = np.zeros((len(all_data), window_size), dtype=np.float64)
    for i, data in enumerate(all_data):
        arr = np.asarray(data, dtype=np.float64).flatten()
        stacked_data[i, : len(arr)] = arr
 
    metadata = pd.DataFrame(all_metadata)
 
    print(f"데이터 로드 완료: {len(all_data)}개 샘플")
    print(f"클래스별 샘플 수:")
    print(metadata["bearing_condition"].value_counts().to_string())
    print(f"\n클래스별 인덱스 범위:")
    for cond in metadata["bearing_condition"].unique():
        idxs = metadata[metadata["bearing_condition"] == cond].index.tolist()
        print(f"  {cond}: {min(idxs)} ~ {max(idxs)} (총 {len(idxs)}개)")
 
    return stacked_data, metadata

In [11]:
# 각 클래스에서 n개의 테스트 샘플을 랜덤 선택
def select_test_samples_by_class(metadata, n_samples_per_class = 1, random_seed = 7):
    random.seed(random_seed)
    np.random.seed(random_seed)
    test_indices = []
    
    for condition in metadata["bearing_condition"].unique():
        class_indices = metadata[
            metadata["bearing_condition"] == condition
        ].index.tolist()
 
        if len(class_indices) >= n_samples_per_class:
            selected = random.sample(class_indices, n_samples_per_class)
            test_indices.extend(selected)
            print(f"  {condition}: {len(selected)}개 선택 (인덱스: {selected})")
        else:
            print(f"  경고: {condition} - 샘플 부족 ({len(class_indices)}개)")
            test_indices.extend(class_indices)
 
    return sorted(test_indices)

In [12]:
# 메인 실행
SAMPLING_RATE = 25600   # Hz
RPM = 1780
DATA_DIR = r".\KAIST_Bearing"

In [13]:
print("데이터 로딩 중")
stacked_data, metadata = load_kaist_data(data_dir=DATA_DIR)

데이터 로딩 중
데이터 로드 완료: 67개 샘플
클래스별 샘플 수:
bearing_condition
normal     23
BPFI_10    11
BPFI_30    11
BPFO_10    11
BPFO_30    11

클래스별 인덱스 범위:
  normal: 0 ~ 22 (총 23개)
  BPFI_10: 23 ~ 33 (총 11개)
  BPFI_30: 34 ~ 44 (총 11개)
  BPFO_10: 45 ~ 55 (총 11개)
  BPFO_30: 56 ~ 66 (총 11개)


In [14]:
print("클래스별 테스트 샘플 선택")
test_indices = select_test_samples_by_class(metadata, n_samples_per_class=1)
print(f"선택된 테스트 인덱스 : {test_indices}")

calculator = BearingFaultCalculator(rpm=RPM, d_in=25, d_out=52, width=15)
fault_frequencies = calculator.calculate_frequencies()

print(f"베어링 결함 주파수 : ")
print(f"    RPM:  {fault_frequencies['RPM']:.0f}")
print(f"    BPFI: {fault_frequencies['BPFI (Inner Race)']:.2f} Hz")
print(f"    BPFO: {fault_frequencies['BPFO (Outer Race)']:.2f} Hz")
print(f"    BSF:  {fault_frequencies['BSF (Ball Spin)']:.2f} Hz")
print(f"    FTF:  {fault_frequencies['FTF (Cage)']:.2f} Hz")

클래스별 테스트 샘플 선택
  normal: 1개 선택 (인덱스: [10])
  BPFI_10: 1개 선택 (인덱스: [25])
  BPFI_30: 1개 선택 (인덱스: [40])
  BPFO_10: 1개 선택 (인덱스: [55])
  BPFO_30: 1개 선택 (인덱스: [56])
선택된 테스트 인덱스 : [10, 25, 40, 55, 56]
베어링 결함 주파수 : 
    RPM:  1780
    BPFI: 161.02 Hz
    BPFO: 105.98 Hz
    BSF:  68.89 Hz
    FTF:  11.78 Hz


In [15]:
print("LLM 모델 로드")
print(f"모델: {MODEL_ID}")
print("FP16 모드")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
qwen_model.eval()

text_generator = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.1,          # 보고서이므로 낮은 temperature
    repetition_penalty=1.2,
    return_full_text=False,
)

if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / (1024 ** 3)
    print(f"모델 로드 완료 (현재 VRAM 사용량: {vram_used:.2f} GB)")

LLM 모델 로드
모델: Qwen/Qwen2.5-3B-Instruct
FP16 모드


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


모델 로드 완료 (현재 VRAM 사용량: 5.75 GB)


In [20]:
print("RAG 기반 AI 진단 보고서 생성")
print(f"테스트 샘플: {len(test_indices)}개")
print("bearing_condition은 진단에 사용하지 않으며, BPFI/BPFO 유사도만 사용\n")

all_reports_data = []

for i, test_idx in enumerate(test_indices, 1):
    print(f"[진단 {i}/{len(test_indices)}] 테스트 샘플 #{test_idx}")

    actual_condition = metadata.iloc[test_idx]["bearing_condition"]
    print(f"참고: 실제 상태 (진단 전): {actual_condition}")

    # 진단 보고서 생성
    report, analysis_data = generate_kaist_report_qwen(
        stacked_data=stacked_data,
        metadata=metadata,
        fault_frequencies=fault_frequencies,
        fs=SAMPLING_RATE,
        idx=test_idx,
        text_generator=text_generator,
        machine_name="KAIST-Bearing",
        use_rag=True,
    )

    # 결과 출력
    print(f"\n[측정 데이터]")
    print(f"  부하: {analysis_data['load']} Nm")
    print(f"  RPM:  {analysis_data['rpm']}")
    print(f"  RMS:  {analysis_data['rms']:.4f}")

    bpfi_val = analysis_data["bpfi_avg_power"]
    bpfo_val = analysis_data["bpfo_avg_power"]
    print(
        f"  BPFI 파워: {bpfi_val:.6e}" if bpfi_val is not None else "  BPFI 파워: N/A"
    )
    print(
        f"  BPFO 파워: {bpfo_val:.6e}" if bpfo_val is not None else "  BPFO 파워: N/A"
    )

    print("[RAG 기반 AI 진단 보고서]")
    print(report)

    all_reports_data.append(
        {
            "Test Index": test_idx,
            "Actual Condition": actual_condition,
            "Load (Nm)": analysis_data["load"],
            "RPM": analysis_data["rpm"],
            "RMS": analysis_data["rms"],
            "BPFI Avg Power (1000-3000Hz)": bpfi_val,
            "BPFO Avg Power (1000-3000Hz)": bpfo_val,
            "Generated Report": report,
        }
    )

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


RAG 기반 AI 진단 보고서 생성
테스트 샘플: 5개
bearing_condition은 진단에 사용하지 않으며, BPFI/BPFO 유사도만 사용

[진단 1/5] 테스트 샘플 #10
참고: 실제 상태 (진단 전): normal
FAISS 의미 검색 — 후보 20개 추출
  쿼리: 베어링 진동 분석 결과: 부하 4.0Nm, RMS 1.5530, BPFI 평균 파워 6.6828e-04, BPFO 평균 파워 2.9396e-05
-> 의미 유사도 범위: 0.8207 ~ 0.8293
수치 오차 리랭킹 — 상위 2개 선정

[검색 결과] 입력: BPFI=6.68e-04, BPFO=2.94e-05
  [1] 오차합: 4.27e+01, 유사도: 0.3781
  [2] 오차합: 4.40e+01, 유사도: 0.3777

[측정 데이터]
  부하: 4.0 Nm
  RPM:  1780
  RMS:  1.5530
  BPFI 파워: 6.682836e-04
  BPFO 파워: 2.939641e-05
[RAG 기반 AI 진단 보고서]
**조치 보고서**

현재 베어링의 상태는 **정상**입니다. 

진동 RMS 값이 1.5530이며 이는 일반적인 범용 기준에 부합합니다. 또한, BPFI 및 BPFO 평균 파워 값이 모두 0.00인 것은 이상 징후 없음을 나타냅니다.

추천하는 정비 액션 아이템은 다음과 같습니다:

1) 센서 체결 상태 및 케이블 단선 또는 노이즈 여부를 확인한다.
2) 정기 점검 주기를 유지하면서 동일 조건에서의 추세를 계속 모니터링한다.

위험도는 **정상**입니다.
[진단 2/5] 테스트 샘플 #25
참고: 실제 상태 (진단 전): BPFI_10
FAISS 의미 검색 — 후보 20개 추출
  쿼리: 베어링 진동 분석 결과: 부하 4.0Nm, RMS 12.9662, BPFI 평균 파워 1.1113e-01, BPFO 평균 파워 2.8225e-04
-> 의미 유사도 범위: 0.8102 ~ 0.8257
수치 오차 리랭킹 — 상위 2개 선정

[검색 결과] 입력: BPFI

In [21]:
print("진단 완료")

reports_df = pd.DataFrame(all_reports_data)
safe_model_id = MODEL_ID.replace("/", "_").replace("-", "_")
output_filename = f"./FaultDiagnosis_reports_{safe_model_id}.csv"
reports_df.to_csv(output_filename, index=False, encoding="utf-8-sig")
print(f"보고서 저장: {output_filename}")

if torch.cuda.is_available():
    vram_peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"최대 VRAM 사용량: {vram_peak:.2f} GB")

진단 완료
보고서 저장: ./FaultDiagnosis_reports_Qwen_Qwen2.5_3B_Instruct.csv
최대 VRAM 사용량: 6.33 GB
